# NGLab Tutorial #7: Reinforcement Learning Training

Train your first deep RL agent using PPO (Proximal Policy Optimization) on the TradingEnv.

## Learning Objectives

1. Configure a PPO agent with Hydra
2. Train for 10k steps
3. Monitor TensorBoard metrics
4. Evaluate and visualize performance

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from collections import deque

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. Understanding PPO

**Proximal Policy Optimization** is a policy gradient method that:
- Uses a clipped objective to prevent large policy updates
- Balances exploration (entropy) with exploitation
- Works well for continuous and discrete action spaces

### The PPO Objective

$$
L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left(r_t(\theta) \hat{A}_t, \ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]
$$

Where:
- $r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{old}}(a_t | s_t)}$ (probability ratio)
- $\hat{A}_t$ = Advantage estimate
- $\epsilon$ = Clip parameter (typically 0.2)

## 2. Simple Policy Network

In [ ]:
class PolicyNetwork(nn.Module):
    """Simple MLP policy for discrete actions."""
    
    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1)
        )
    
    def forward(self, x):
        return self.network(x)

class ValueNetwork(nn.Module):
    """Critic network for value estimation."""
    
    def __init__(self, obs_dim: int, hidden_dim: int = 256):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.network(x).squeeze(-1)

# Create networks
obs_dim = 60  # Lookback window
action_dim = 3  # Hold, Buy, Sell

policy = PolicyNetwork(obs_dim, action_dim)
value_fn = ValueNetwork(obs_dim)

print(f"Policy network parameters: {sum(p.numel() for p in policy.parameters()):,}")
print(f"Value network parameters: {sum(p.numel() for p in value_fn.parameters()):,}")

## 3. Mock Training Loop

Since full PPO training requires significant compute, we'll demonstrate a simplified version:

In [ ]:
# Hyperparameters
gamma = 0.99
lambd = 0.95
clip_epsilon = 0.2
learning_rate = 3e-4
n_epochs = 4
batch_size = 64

# Optimizers
policy_optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)
value_optimizer = torch.optim.Adam(value_fn.parameters(), lr=learning_rate)

print("\n=== Training Configuration ===")
print(f"Discount factor (γ): {gamma}")
print(f"GAE lambda (λ): {lambd}")
print(f"Clip epsilon (ε): {clip_epsilon}")
print(f"Learning rate: {learning_rate}")
print(f"Epochs per update: {n_epochs}")

In [ ]:
# Simulate training metrics
np.random.seed(42)
n_updates = 100

metrics = {
    'policy_loss': [],
    'value_loss': [],
    'entropy': [],
    'avg_reward': [],
    'portfolio_value': []
}

# Simulate improving performance
for i in range(n_updates):
    # Policy loss should decrease
    policy_loss = 0.5 * np.exp(-i/20) + 0.01 + np.random.randn() * 0.01
    
    # Value loss should decrease
    value_loss = 0.3 * np.exp(-i/25) + 0.005 + np.random.randn() * 0.005
    
    # Entropy should stay relatively constant (exploration)
    entropy = 1.1 - 0.3 * (i/n_updates) + np.random.randn() * 0.05
    
    # Rewards should improve
    avg_reward = -0.01 + 0.03 * (i/n_updates) + np.random.randn() * 0.005
    
    # Portfolio value trending up
    portfolio_value = 100000 * (1 + 0.1 * (i/n_updates)) + np.random.randn() * 500
    
    metrics['policy_loss'].append(policy_loss)
    metrics['value_loss'].append(value_loss)
    metrics['entropy'].append(entropy)
    metrics['avg_reward'].append(avg_reward)
    metrics['portfolio_value'].append(portfolio_value)

print("✓ Training simulation complete (100 updates)")

## 4. Visualizing Training Progress

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Policy loss
axes[0, 0].plot(metrics['policy_loss'], color='blue', linewidth=2)
axes[0, 0].set_title('Policy Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Update')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# Value loss
axes[0, 1].plot(metrics['value_loss'], color='orange', linewidth=2)
axes[0, 1].set_title('Value Function Loss', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Update')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].grid(True, alpha=0.3)

# Average reward
axes[1, 0].plot(metrics['avg_reward'], color='green', linewidth=2)
axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Average Episode Reward', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Update')
axes[1, 0].set_ylabel('Reward')
axes[1, 0].grid(True, alpha=0.3)

# Portfolio value
axes[1, 1].plot(metrics['portfolio_value'], color='purple', linewidth=2)
axes[1, 1].axhline(y=100000, color='red', linestyle='--', label='Initial', alpha=0.5)
axes[1, 1].set_title('Portfolio Value', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Update')
axes[1, 1].set_ylabel('Value (USD)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. TensorBoard Integration

In production training, you would log metrics to TensorBoard:

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter('runs/ppo_trading')

for step in range(total_steps):
    # ... training code ...
    
    writer.add_scalar('loss/policy', policy_loss, step)
    writer.add_scalar('loss/value', value_loss, step)
    writer.add_scalar('metrics/reward', avg_reward, step)
    writer.add_scalar('metrics/entropy', entropy, step)

Then visualize with:
```bash
tensorboard --logdir runs/
```

## 6. Entropy Analysis

Entropy measures policy randomness. Too low = exploitation, too high = random.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(metrics['entropy'], linewidth=2, color='teal')
plt.axhline(y=1.1, color='green', linestyle='--', alpha=0.5, label='Max entropy (uniform)')
plt.title('Policy Entropy Over Training', fontsize=14, fontweight='bold')
plt.xlabel('Update')
plt.ylabel('Entropy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nEntropy statistics:")
print(f"  Initial: {metrics['entropy'][0]:.3f}")
print(f"  Final: {metrics['entropy'][-1]:.3f}")
print(f"  Mean: {np.mean(metrics['entropy']):.3f}")

## 7. Running Real Training

To train a real agent, use the NGLab CLI:

```bash
# Basic PPO training
python python/src/main.py algorithm=ppo task=trading model=tsmamba

# With custom hyperparameters
python python/src/main.py \
    algorithm=ppo \
    task=trading \
    model=tsmamba \
    algorithm.learning_rate=1e-4 \
    algorithm.gamma=0.99 \
    algorithm.n_steps=2048
```

Monitor progress:
```bash
tensorboard --logdir outputs/
```

## Summary

In this notebook, you learned:

✅ PPO algorithm fundamentals  
✅ Policy and value network architectures  
✅ Training metrics visualization  
✅ TensorBoard integration  
✅ How to launch real training jobs  

## Next Steps

Continue to **Notebook #8**: Multi-Agent Simulation for competitive scenarios!

---